 # 02 · Ejecutar Data Quality Bronze — YelpConnect

 **Dominio:** Data Quality  
 **Capa:** Bronze  
 **Tipo:** Ejecución / Quality Gate  
 **Origen de reglas:** `workspace.yelp_gov.dq_reglas`  
 **Destino de resultados:** `workspace.yelp_gov.dq_resultados`

 ---

 ## Propósito

 Ejecutar las reglas de calidad previamente publicadas para Bronze
 y determinar si el pipeline puede continuar hacia Silver.

 ### Principio

 **Bronze conserva; DQ observa; Silver corrige.**

 Este notebook:

 - valida precondiciones;
 - lee el catálogo DQ Bronze;
 - ejecuta las reglas;
 - calcula métricas;
 - registra resultados históricos;
 - genera scorecard;
 - aplica el Quality Gate.

 Este notebook **NO**:

 - crea schemas;
 - crea tablas de gobierno;
 - publica reglas;
 - modifica datos Bronze;
 - elimina o corrige registros.

 ---

 ## Política del Quality Gate

 - Una regla de criticidad **alta** incumplida bloquea el pipeline.
 - Una regla de criticidad **media** se registra, pero no bloquea.
 - Los resultados quedan almacenados en `dq_resultados`.

In [0]:

# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

import json
import uuid

from datetime import (
    datetime,
    timezone
)

from pyspark.sql import functions as F


In [0]:

# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "yelp_bronze"

GOV_SCHEMA = "yelp_gov"

LAYER = "bronze"

PIPELINE_NAME = (
    "E5_02_ejecutar_dq_bronze"
)


DQ_RULE_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_reglas"
)


DQ_RESULT_TABLE = (
    f"{CATALOG}."
    f"{GOV_SCHEMA}."
    "dq_resultados"
)


DQ_EXECUTION_ID = str(
    uuid.uuid4()
)


DQ_EXECUTION_TS = (
    datetime.now(
        timezone.utc
    )
)


EXPECTED_RULE_COUNT = 22


EXPECTED_RULE_IDS = {
    f"DQ-BR-{number:03d}"
    for number
    in range(
        1,
        EXPECTED_RULE_COUNT + 1
    )
}


EXPECTED_BRONZE_TABLES = {

    f"{CATALOG}.{BRONZE_SCHEMA}.brz_business",

    f"{CATALOG}.{BRONZE_SCHEMA}.brz_user",

    f"{CATALOG}.{BRONZE_SCHEMA}.brz_review",

    f"{CATALOG}.{BRONZE_SCHEMA}.brz_tip",

    f"{CATALOG}.{BRONZE_SCHEMA}.brz_checkin"
}


print("=" * 72)
print("YELPCONNECT - DATA QUALITY BRONZE")
print("=" * 72)

print(
    f"Pipeline      : {PIPELINE_NAME}"
)

print(
    f"Execution ID  : {DQ_EXECUTION_ID}"
)

print(
    f"Execution UTC : {DQ_EXECUTION_TS.isoformat()}"
)

print(
    f"Layer         : {LAYER}"
)

print(
    f"Rule catalog  : {DQ_RULE_TABLE}"
)

print(
    f"Results       : {DQ_RESULT_TABLE}"
)

print("=" * 72)


# COMMAND ----------

 ## 1 · Validación de precondiciones

 Gobierno e ingesta Bronze deben existir antes de ejecutar DQ.

 Ante un incumplimiento de contrato, el notebook falla inmediatamente
 en lugar de crear o reparar infraestructura silenciosamente.

In [0]:

# COMMAND ----------

# ============================================================
# 3. FUNCIÓN DE VALIDACIÓN DE CONTRATOS
# ============================================================

def assert_table_contract(
    table_name,
    expected_contract
):

    if not spark.catalog.tableExists(
        table_name
    ):

        raise RuntimeError(
            f"""
            ❌ TABLA REQUERIDA INEXISTENTE

            Tabla:
            {table_name}
            """
        )


    actual_contract = {

        field.name:
            field.dataType
            .simpleString()
            .lower()

        for field
        in spark.table(
            table_name
        ).schema.fields
    }


    missing_columns = [

        column_name

        for column_name
        in expected_contract

        if column_name
        not in actual_contract
    ]


    if missing_columns:

        raise RuntimeError(
            f"""
            ❌ CONTRATO INCOMPLETO

            Tabla:
            {table_name}

            Columnas faltantes:
            {missing_columns}
            """
        )


    invalid_types = []


    for (
        column_name,
        expected_type
    ) in expected_contract.items():

        actual_type = (
            actual_contract[
                column_name
            ]
        )

        if (
            actual_type
            !=
            expected_type.lower()
        ):

            invalid_types.append({
                "column":
                    column_name,

                "expected":
                    expected_type,

                "actual":
                    actual_type
            })


    if invalid_types:

        raise RuntimeError(
            f"""
            ❌ CONTRATO CON TIPOS INCOMPATIBLES

            Tabla:
            {table_name}

            Detalle:
            {invalid_types}
            """
        )


    print(
        f"✔ Contrato válido: "
        f"{table_name}"
    )



In [0]:


# COMMAND ----------

# ============================================================
# 4. CONTRATO dq_reglas
# ============================================================

DQ_RULE_CONTRACT = {

    "regla_id":
        "string",

    "capa":
        "string",

    "tabla_objetivo":
        "string",

    "columna":
        "string",

    "dimension":
        "string",

    "tipo_regla":
        "string",

    "descripcion":
        "string",

    "expresion_sql":
        "string",

    "criticidad":
        "string",

    "umbral_pct":
        "double"
}


assert_table_contract(
    DQ_RULE_TABLE,
    DQ_RULE_CONTRACT
)



In [0]:

# COMMAND ----------

# ============================================================
# 5. CONTRATO dq_resultados
# ============================================================

DQ_RESULT_CONTRACT = {

    "ejecucion_id":
        "string",

    "ejecucion_ts":
        "timestamp",

    "regla_id":
        "string",

    "capa":
        "string",

    "tabla_objetivo":
        "string",

    "columna":
        "string",

    "dimension":
        "string",

    "tipo_regla":
        "string",

    "criticidad":
        "string",

    "filas_totales":
        "bigint",

    "filas_ok":
        "bigint",

    "filas_error":
        "bigint",

    "pct_cumplimiento":
        "double",

    "umbral_pct":
        "double",

    "paso":
        "boolean"
}


assert_table_contract(
    DQ_RESULT_TABLE,
    DQ_RESULT_CONTRACT
)




In [0]:

# COMMAND ----------

# ============================================================
# 6. VALIDAR TABLAS BRONZE
# ============================================================

missing_bronze_tables = [

    table_name

    for table_name
    in sorted(
        EXPECTED_BRONZE_TABLES
    )

    if not spark.catalog.tableExists(
        table_name
    )
]


if missing_bronze_tables:

    raise RuntimeError(
        f"""
        ❌ INGESTA BRONZE INCOMPLETA

        Tablas faltantes:

        {missing_bronze_tables}

        Ejecutar previamente:
        E7_pipeline_lakehouse/01_ingestar_bronze
        """
    )


print(
    "✔ Las 5 tablas Bronze "
    "se encuentran disponibles."
)


# COMMAND ----------

 ## 2 · Cargar catálogo DQ Bronze

 El executor no contiene definiciones hardcodeadas de las reglas.

 Las reglas deben haber sido publicadas previamente mediante:

 `00_publicar_catalogo_reglas_dq_bronze`


In [0]:

# COMMAND ----------

# ============================================================
# 7. CARGAR REGLAS BRONZE
# ============================================================

df_rules = (

    spark

    .table(
        DQ_RULE_TABLE
    )

    .filter(
        F.col(
            "capa"
        )
        ==
        LAYER
    )

    .orderBy(
        "regla_id"
    )
)


actual_rule_count = (
    df_rules.count()
)


if (
    actual_rule_count
    !=
    EXPECTED_RULE_COUNT
):

    raise RuntimeError(
        f"""
        ❌ CANTIDAD DE REGLAS BRONZE INCORRECTA

        Esperadas:
        {EXPECTED_RULE_COUNT}

        Encontradas:
        {actual_rule_count}

        Ejecutar previamente:
        00_publicar_catalogo_reglas_dq_bronze
        """
    )


print(
    f"✔ Reglas Bronze cargadas: "
    f"{actual_rule_count}"
)



In [0]:

# COMMAND ----------

# ============================================================
# 8. VALIDAR IDS CANÓNICOS
# ============================================================

actual_rule_ids = {

    row.regla_id

    for row
    in df_rules
        .select(
            "regla_id"
        )
        .collect()
}


missing_rule_ids = (
    EXPECTED_RULE_IDS
    -
    actual_rule_ids
)


extra_rule_ids = (
    actual_rule_ids
    -
    EXPECTED_RULE_IDS
)


if (
    missing_rule_ids
    or
    extra_rule_ids
):

    raise RuntimeError(
        f"""
        ❌ CATÁLOGO BRONZE NO CANÓNICO

        IDs faltantes:
        {sorted(missing_rule_ids)}

        IDs adicionales:
        {sorted(extra_rule_ids)}
        """
    )


print(
    "✔ IDs DQ-BR-001 ... DQ-BR-022 válidos."
)


In [0]:


# COMMAND ----------

# ============================================================
# 9. VALIDAR DUPLICADOS
# ============================================================

duplicate_rule_ids = (

    df_rules

    .groupBy(
        "regla_id"
    )

    .count()

    .filter(
        F.col(
            "count"
        )
        >
        1
    )

    .collect()
)


if duplicate_rule_ids:

    raise RuntimeError(
        f"""
        ❌ EXISTEN REGLAS DUPLICADAS

        {duplicate_rule_ids}
        """
    )


print(
    "✔ No existen regla_id duplicados."
)


In [0]:

# ============================================================
# 10. VALIDAR TIPOS SOPORTADOS
# ============================================================

SUPPORTED_RULE_TYPES = {
    "TABLE_NOT_EMPTY",
    "CONDITION",
    "UNIQUENESS"
}


invalid_rule_types = (

    df_rules

    .filter(
        ~F.col(
            "tipo_regla"
        ).isin(
            list(
                SUPPORTED_RULE_TYPES
            )
        )
    )

    .select(
        "regla_id",
        "tipo_regla"
    )

    .collect()
)


if invalid_rule_types:

    raise RuntimeError(
        f"""
        ❌ TIPO DE REGLA NO SOPORTADO

        {invalid_rule_types}
        """
    )


print(
    "✔ Tipos de regla soportados."
)


In [0]:

# COMMAND ----------

# ============================================================
# 11. VALIDAR TABLAS OBJETIVO DEL CATÁLOGO
# ============================================================

catalog_target_tables = {

    row.tabla_objetivo

    for row
    in df_rules
        .select(
            "tabla_objetivo"
        )
        .distinct()
        .collect()
}


if (
    catalog_target_tables
    !=
    EXPECTED_BRONZE_TABLES
):

    raise RuntimeError(
        f"""
        ❌ TABLAS OBJETIVO DEL CATÁLOGO INCORRECTAS

        Esperadas:
        {sorted(EXPECTED_BRONZE_TABLES)}

        Encontradas:
        {sorted(catalog_target_tables)}
        """
    )


print(
    "✔ Catálogo apunta a las 5 tablas Bronze esperadas."
)


# COMMAND ----------


 ## 3 · Motor genérico de Data Quality

 El motor soporta:

 - `TABLE_NOT_EMPTY`
 - `CONDITION`
 - `UNIQUENESS`

 Cada resultado conserva:

 - regla;
 - tabla;
 - columna;
 - dimensión;
 - tipo de regla;
 - criticidad;
 - total;
 - filas válidas;
 - filas con error;
 - porcentaje;
 - umbral;
 - resultado PASS/FAIL.


In [0]:

# COMMAND ----------

# ============================================================
# 12. EJECUTAR REGLAS
# ============================================================

rules = (
    df_rules.collect()
)


results = []


table_cache = {}

table_counts = {}


for rule in rules:

    print()
    print("-" * 72)

    print(
        f"Regla       : {rule.regla_id}"
    )

    print(
        f"Descripción : {rule.descripcion}"
    )

    print(
        f"Tabla       : {rule.tabla_objetivo}"
    )

    print(
        f"Tipo        : {rule.tipo_regla}"
    )


    # ========================================================
    # OBTENER TABLA
    # ========================================================

    if (
        rule.tabla_objetivo
        not in table_cache
    ):

        table_cache[
            rule.tabla_objetivo
        ] = spark.table(
            rule.tabla_objetivo
        )


    df = (
        table_cache[
            rule.tabla_objetivo
        ]
    )


    # ========================================================
    # TOTAL DE REGISTROS
    # ========================================================

    if (
        rule.tabla_objetivo
        not in table_counts
    ):

        table_counts[
            rule.tabla_objetivo
        ] = df.count()


    total = (
        table_counts[
            rule.tabla_objetivo
        ]
    )


    # ========================================================
    # TABLE_NOT_EMPTY
    # ========================================================

    if (
        rule.tipo_regla
        ==
        "TABLE_NOT_EMPTY"
    ):

        if total > 0:

            filas_ok = total

        else:

            filas_ok = 0


    # ========================================================
    # UNIQUENESS
    # ========================================================

    elif (
        rule.tipo_regla
        ==
        "UNIQUENESS"
    ):

        if (
            rule.columna
            not in df.columns
        ):

            raise RuntimeError(
                f"""
                ❌ COLUMNA INEXISTENTE

                Regla:
                {rule.regla_id}

                Tabla:
                {rule.tabla_objetivo}

                Columna:
                {rule.columna}
                """
            )


        unique_count = (

            df

            .agg(
                F.countDistinct(
                    F.col(
                        rule.columna
                    )
                ).alias(
                    "unique_count"
                )
            )

            .first()
            .unique_count
        )


        filas_ok = (
            unique_count
        )


    # ========================================================
    # CONDITION
    # ========================================================

    elif (
        rule.tipo_regla
        ==
        "CONDITION"
    ):

        if (
            rule.expresion_sql
            is None
            or
            not rule.expresion_sql.strip()
        ):

            raise RuntimeError(
                f"""
                ❌ CONDITION SIN EXPRESIÓN

                Regla:
                {rule.regla_id}
                """
            )


        try:

            filas_ok = (

                df

                .filter(
                    F.expr(
                        rule.expresion_sql
                    )
                )

                .count()
            )

        except Exception as exc:

            raise RuntimeError(
                f"""
                ❌ ERROR EJECUTANDO CONDITION

                Regla:
                {rule.regla_id}

                Tabla:
                {rule.tabla_objetivo}

                Expresión:
                {rule.expresion_sql}
                """
            ) from exc


    # ========================================================
    # TIPO NO SOPORTADO
    # ========================================================

    else:

        raise RuntimeError(
            f"""
            ❌ TIPO DE REGLA NO SOPORTADO

            Regla:
            {rule.regla_id}

            Tipo:
            {rule.tipo_regla}
            """
        )


    # ========================================================
    # MÉTRICAS
    # ========================================================

    filas_error = (
        total
        -
        filas_ok
    )


    pct_cumplimiento = (

        round(
            (
                100.0
                *
                filas_ok
                /
                total
            ),
            2
        )

        if total > 0

        else 0.0
    )


    paso = (
        pct_cumplimiento
        >=
        rule.umbral_pct
    )


    # ========================================================
    # RESULTADO
    # ========================================================

    results.append(
        (
            DQ_EXECUTION_ID,
            DQ_EXECUTION_TS,
            rule.regla_id,
            rule.capa,
            rule.tabla_objetivo,
            rule.columna,
            rule.dimension,
            rule.tipo_regla,
            rule.criticidad,
            total,
            filas_ok,
            filas_error,
            pct_cumplimiento,
            rule.umbral_pct,
            paso
        )
    )


    status_icon = (
        "✔"
        if paso
        else "❌"
    )


    print(
        f"{status_icon} "
        f"{pct_cumplimiento}% "
        f"| umbral={rule.umbral_pct}% "
        f"| error={filas_error:,}"
    )


# COMMAND ----------


 ## 4 · Construir resultados

 El contrato utilizado aquí debe coincidir exactamente con
 `workspace.yelp_gov.dq_resultados`.

In [0]:

# COMMAND ----------

# ============================================================
# 13. CREAR DATAFRAME DE RESULTADOS
# ============================================================

RESULT_SCHEMA = """
    ejecucion_id STRING,
    ejecucion_ts TIMESTAMP,
    regla_id STRING,
    capa STRING,
    tabla_objetivo STRING,
    columna STRING,
    dimension STRING,
    tipo_regla STRING,
    criticidad STRING,
    filas_totales BIGINT,
    filas_ok BIGINT,
    filas_error BIGINT,
    pct_cumplimiento DOUBLE,
    umbral_pct DOUBLE,
    paso BOOLEAN
"""


df_results = (
    spark.createDataFrame(
        results,
        RESULT_SCHEMA
    )
)


result_count = (
    df_results.count()
)


if (
    result_count
    !=
    EXPECTED_RULE_COUNT
):

    raise RuntimeError(
        f"""
        ❌ RESULTADOS DQ INCOMPLETOS

        Esperados:
        {EXPECTED_RULE_COUNT}

        Generados:
        {result_count}
        """
    )


print(
    f"✔ Resultados generados: "
    f"{result_count}"
)


In [0]:

# COMMAND ----------

# ============================================================
# 14. PERSISTIR RESULTADOS HISTÓRICOS
# ============================================================

RESULT_COLUMNS = [
    "ejecucion_id",
    "ejecucion_ts",
    "regla_id",
    "capa",
    "tabla_objetivo",
    "columna",
    "dimension",
    "tipo_regla",
    "criticidad",
    "filas_totales",
    "filas_ok",
    "filas_error",
    "pct_cumplimiento",
    "umbral_pct",
    "paso"
]


(
    df_results

    .select(
        *RESULT_COLUMNS
    )

    .write

    .format(
        "delta"
    )

    .mode(
        "append"
    )

    .saveAsTable(
        DQ_RESULT_TABLE
    )
)


print(
    f"✔ Resultados persistidos: "
    f"{DQ_RESULT_TABLE}"
)


# COMMAND ----------


## 5 · Resultados de la ejecución


In [0]:
 
# COMMAND ----------

# ============================================================
# 15. DETALLE
# ============================================================

display(
    df_results
    .orderBy(
        F.col(
            "paso"
        ).asc(),
        F.col(
            "criticidad"
        ).asc(),
        F.col(
            "regla_id"
        ).asc()
    )
)


In [0]:

# COMMAND ----------

# ============================================================
# 16. SCORECARD POR DIMENSIÓN
# ============================================================

df_scorecard = (

    df_results

    .groupBy(
        "dimension"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "reglas"
        ),

        F.sum(
            F.when(
                F.col(
                    "paso"
                ),
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "reglas_pass"
        ),

        F.sum(
            F.when(
                ~F.col(
                    "paso"
                ),
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "reglas_fail"
        ),

        F.round(
            F.avg(
                "pct_cumplimiento"
            ),
            2
        ).alias(
            "cumplimiento_promedio"
        )
    )

    .orderBy(
        "dimension"
    )
)


display(
    df_scorecard
)



In [0]:
# COMMAND ----------

# ============================================================
# 17. FALLAS
# ============================================================

df_failures = (

    df_results

    .filter(
        ~F.col(
            "paso"
        )
    )

    .orderBy(
        "criticidad",
        "regla_id"
    )
)


display(
    df_failures
)


# COMMAND ----------


 ## 6 · Quality Gate Bronze

 Una falla de criticidad **alta** bloquea el paso hacia Silver.

 Las fallas de criticidad media quedan registradas para seguimiento,
 pero no detienen la ejecución.

In [0]:

# COMMAND ----------

# ============================================================
# 18. MÉTRICAS DEL QUALITY GATE
# ============================================================

rules_passed = (

    df_results

    .filter(
        F.col(
            "paso"
        )
    )

    .count()
)


rules_failed = (
    EXPECTED_RULE_COUNT
    -
    rules_passed
)


critical_failures = (

    df_results

    .filter(
        (
            F.col(
                "criticidad"
            )
            ==
            "alta"
        )

        &

        (
            ~F.col(
                "paso"
            )
        )
    )

    .count()
)


medium_failures = (

    df_results

    .filter(
        (
            F.col(
                "criticidad"
            )
            ==
            "media"
        )

        &

        (
            ~F.col(
                "paso"
            )
        )
    )

    .count()
)


quality_gate_status = (

    "FAIL"

    if critical_failures > 0

    else "PASS"
)



In [0]:

# COMMAND ----------

# ============================================================
# 19. RESUMEN FINAL
# ============================================================

summary = {

    "status":
        (
            "SUCCESS"
            if quality_gate_status == "PASS"
            else "FAILED"
        ),

    "pipeline":
        PIPELINE_NAME,

    "execution_id":
        DQ_EXECUTION_ID,

    "layer":
        LAYER,

    "rules_evaluated":
        EXPECTED_RULE_COUNT,

    "rules_passed":
        rules_passed,

    "rules_failed":
        rules_failed,

    "critical_failures":
        critical_failures,

    "medium_failures":
        medium_failures,

    "quality_gate":
        quality_gate_status,

    "result_table":
        DQ_RESULT_TABLE
}


print()
print("=" * 72)
print("QUALITY GATE BRONZE")
print("=" * 72)

print(
    f"Reglas evaluadas : "
    f"{EXPECTED_RULE_COUNT}"
)

print(
    f"PASS             : "
    f"{rules_passed}"
)

print(
    f"FAIL             : "
    f"{rules_failed}"
)

print(
    f"Fallas altas     : "
    f"{critical_failures}"
)

print(
    f"Fallas medias    : "
    f"{medium_failures}"
)

print(
    f"QUALITY GATE     : "
    f"{quality_gate_status}"
)

print("=" * 72)

print(
    json.dumps(
        summary,
        indent=2
    )
)


# COMMAND ----------

 ## Definition of Done

 - [x] Contrato `dq_reglas` validado.
 - [x] Contrato `dq_resultados` validado.
 - [x] Cinco tablas Bronze validadas.
 - [x] 22 reglas Bronze recuperadas desde gobierno.
 - [x] IDs canónicos validados.
 - [x] Motor genérico DQ ejecutado.
 - [x] `columna` persistida.
 - [x] `tipo_regla` persistido.
 - [x] Resultados históricos persistidos.
 - [x] Scorecard generado.
 - [x] Quality Gate aplicado.
 - [x] Ningún registro Bronze fue modificado.


In [0]:

# COMMAND ----------

# ============================================================
# 20. FINALIZAR EJECUCIÓN
# ============================================================

if critical_failures > 0:

    raise RuntimeError(
        f"""
        ❌ QUALITY GATE BRONZE FALLÓ

        Execution ID:
        {DQ_EXECUTION_ID}

        Reglas evaluadas:
        {EXPECTED_RULE_COUNT}

        Fallas críticas:
        {critical_failures}

        El pipeline NO debe continuar hacia Silver.

        Consultar:
        {DQ_RESULT_TABLE}
        """
    )


print(
    "✔ Quality Gate Bronze superado."
)

print(
    "✔ El pipeline puede continuar hacia Silver."
)


dbutils.notebook.exit(
    json.dumps(
        summary
    )
)